# 08 · Astra: evidence in, reviewable briefing out

**Mission:** prepare a real GPT-6 Astra Responses API request, validate citations, and practice evaluating an AI draft. Allow 45 minutes. Prerequisites: Labs 04–05.

**Run:** choose the Python (Pyodide) kernel in JupyterLite, then Run → Run All Cells. First use downloads Matplotlib. All training data are **SYNTHETIC** unless explicitly labeled LIVE. Downloads appear below and in `exports/`; the browser filesystem is separate from your computer. Each lab runs independently.

In [1]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install("matplotlib")
import json, math
from pathlib import Path
from IPython.display import display, HTML
import matplotlib.pyplot as plt
from worldview_lab import (load, export, map_layer, chart_style, haversine_km,
    wilson_interval, shortest_times, freshness, normalize_quakes,
    validate_geojson, brief_request, validate_brief)
chart_style()

## Capability and responsibility

Official documentation lists reasoning, coding, image input, function calling, and structured outputs for GPT-6 Astra. Here we apply them to map interpretation, code review, and evidence synthesis. Precise spatial calculations stay in tested Python/Cesium code. Image interpretation is fallible; no geospatial accuracy guarantee is assumed.

The default notebook makes **no API call**. It creates a request and tests a deliberately authored fixture. The optional local runner in `tools/astra_brief.py` sends the evidence to `gpt-6-astra` using your server-side environment key. Never paste that key into JupyterLite, committed notebooks, a browser form, or a `VITE_*` variable.

In [2]:
districts=load('districts.geojson');rows=[f['properties'] for f in districts['features']]
top=max(rows,key=lambda r:r['rate_per_100k'])
evidence={'classification':'SYNTHETIC EXERCISE','scenario_clock':'2026-09-14T12:00:00Z',
 'evidence':[{'id':'EPI-01','metric':'total_cases','value':sum(r['cases'] for r in rows),'unit':'new cases / 14 days'},
             {'id':'EPI-02','metric':'highest_scaled_proportion','district':top['id'],'value':top['rate_per_100k'],'unit':'14-day cases per 100,000'}],
 'limitations':['Invented population and cases','No causal inference','No individual-level risk inference']}
request=brief_request(evidence)
print('Model:',request['model'],'| store:',request['store'],'| strict schema:',request['text']['format']['strict'])
export('astra-evidence.json',evidence)
export('astra-request.json',request)

Model: gpt-6-astra | store: False | strict schema: True


WindowsPath('exports/astra-request.json')

In [3]:
fixture={'summary':'SYNTHETIC EXERCISE: review the district comparison against reporting completeness before briefing.',
 'evidence_ids':['EPI-01','EPI-02'],
 'limitations':['Illustrative data do not describe a real outbreak.'],
 'review_questions':['Are time window and denominator consistent?'], 'requires_human_review':True}
assert validate_brief(fixture,evidence)
print('Authored fixture passed structural and citation checks; this is NOT an Astra response.')
bad=dict(fixture,evidence_ids=['INVENTED-SOURCE'])
try:validate_brief(bad,evidence)
except ValueError as error:print('Expected rejection:',error)

Authored fixture passed structural and citation checks; this is NOT an Astra response.
Expected rejection: Unknown or missing evidence citation


## Run Astra outside the browser

Download `astra-evidence.json`. In a local checkout, set `OPENAI_API_KEY` in the process environment through your normal secret-management method, then run:

```bash
python tools/astra_brief.py --evidence path/to/astra-evidence.json --output artifacts/astra-brief.json
```

Optional `--image path/to/figure.png` adds your reviewed map/chart image. `--dry-run` writes the request without contacting OpenAI. The runner rejects missing citations, refusals, incomplete responses, and malformed output. It records request model, response ID, usage, and evidence hash for review. Model/account access and usage charges apply; no inference runs in GitHub Actions.

The schema does not prove truth. Compare every number and statement to the evidence, check causal language, and reject unjustified precision.

In [4]:
# A narrow, deterministic tool dispatcher for teaching tool contracts.
def dispatch(name,arguments):
    if name!='distance_km':raise ValueError('Tool is not allowlisted')
    if set(arguments)!={'from_lon','from_lat','to_lon','to_lat'}:raise ValueError('Unexpected arguments')
    return haversine_km(arguments['from_lon'],arguments['from_lat'],arguments['to_lon'],arguments['to_lat'])
tool_result=dispatch('distance_km',{'from_lon':-77.23,'from_lat':38.71,'to_lon':-77.10,'to_lat':38.83})
print('Local tool result (not a model call):',round(tool_result,2),'km')

Local tool result (not a model call): 17.47 km


## Evaluation lab

Score a draft on five dimensions: correct units, valid evidence IDs, no invented facts, explicit limitations, and useful review questions. Add an adversarial evidence string asking the model to ignore instructions; keep it treated as data. **Checkpoint:** the citation validator rejects an unknown ID but cannot detect every false claim. The dispatcher never executes arbitrary Python suggested by a model.

**Prompt experiments:** ask Astra to explain a choropleth to a nontechnical coordinator; critique a denominator calculation; compare a 2D map with a 3D scene; draft a bilingual handoff with the same evidence IDs. Evaluate each result with a domain reviewer.

## Sources and attribution

[GPT-6 Astra model](https://developers.openai.com/api/docs/models/gpt-6-astra), [Structured Outputs](https://developers.openai.com/api/docs/guides/structured-outputs), [Images and vision](https://developers.openai.com/api/docs/guides/images-vision), and [Function calling](https://developers.openai.com/api/docs/guides/function-calling). Documentation checked 2026-10-02. Exact spatial localization from images remains a documented limitation.

World View inspiration: [Kevin (Khoa) To, MIT](https://github.com/kevtoe/worldview). Original lab code, figures, and synthetic fixtures: JupyterLite WorldView contributors, MIT. See the [book references](https://jltobias.github.io/JupyterLite-WorldView/book/references.html) and repository notices for dependency and service terms.